# 不正検知① — ベースライン：不均衡分類のベースラインをテストで測る

出典：第02章 02節／パス：`02_fraud/01_baseline.ipynb`／パッケージ：numpy・pandas・scikit-learn／実行時間の目安：約1分

このノート単体で実行できる。`input/transactions.csv`（合成の業務らしい取引データ）を読み込む。欠品時は内蔵生成へ切り替わる。
改善モデルのコードは `02_improved.ipynb` に分けてある。最終評価はテストセットで1回だけ行う。


In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RS = 0
rng = np.random.default_rng(7)
N = 20000


def make_fraud_data(n: int, seed: int = 7) -> pd.DataFrame:
    r = np.random.default_rng(seed)
    amount = np.clip(r.lognormal(mean=3.0, sigma=1.0, size=n), 1.0, 500.0)  # 利用金額（千円）
    hour = r.integers(0, 24, size=n)
    night = ((hour <= 4) | (hour >= 23)).astype(float)
    intl = (r.uniform(size=n) < 0.08).astype(int)  # 海外利用
    card_age = np.clip(r.exponential(scale=24.0, size=n), 0.0, 120.0)  # カード経過月数
    new_card = (card_age < 3.0).astype(float)
    online = (r.uniform(size=n) < 0.60).astype(int)
    merchant = r.choice(["retail", "food", "travel", "digital", "other"], size=n,
                       p=[0.35, 0.25, 0.10, 0.15, 0.15])
    risky_merch = np.isin(merchant, ["travel", "digital"]).astype(float)
    # 不正は「高額×深夜×海外×新規カード」の重なりで起きる（交互作用が支配的）
    logit = (-5.2 + 0.030 * amount * night + 3.0 * intl * new_card
             + 2.2 * risky_merch * online + 1.6 * new_card * night
             + 0.008 * amount * intl - 0.02 * card_age)
    prob = 1.0 / (1.0 + np.exp(-logit))
    fraud = (r.uniform(size=n) < prob).astype(int)
    return pd.DataFrame({
        "amount": amount, "hour": hour, "intl": intl, "card_age": card_age,
        "online": online, "merchant": merchant, "fraud": fraud,
    })


# Kaggle ULB版（creditcard.csv）を同フォルダに置くと公開データで動く。無ければInput、さらに無ければ内蔵生成
PUBLIC_CSV = "creditcard.csv"
INPUT = "input/transactions.csv"
try:
    df = pd.read_csv(PUBLIC_CSV).rename(columns={"Class": "fraud"})
    num_cols = ["Amount"] + [f"V{i}" for i in range(1, 29)]
    cat_cols = []
    print(f"公開データを使用: {PUBLIC_CSV}")
except FileNotFoundError:
    num_cols = ["amount", "hour", "intl", "card_age", "online"]
    cat_cols = ["merchant"]
    try:
        df = pd.read_csv(INPUT)
        print(f"Inputを使用: {INPUT}")
    except FileNotFoundError:
        df = make_fraud_data(N)
        print("Inputが無いため合成データを生成")
print(f"件数: {len(df)}, 不正率: {df['fraud'].mean():.4f}")

Inputを使用: input/transactions.csv
件数: 20000, 不正率: 0.0192


In [2]:
X = df.drop(columns=["fraud"])
y = df["fraud"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RS
)
pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
print(f"学習: {len(X_tr)}件 / テスト: {len(X_te)}件（テストの不正 {int(y_te.sum())}件）")

学習: 15000件 / テスト: 5000件（テストの不正 96件）


In [3]:
# ベースライン①：陽性率で振る物差し（最終評価はテストで1回だけ）
dummy = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
p_dummy = dummy.predict_proba(X_te)[:, 1]
print(f"[ダミー] PR-AUC={average_precision_score(y_te, p_dummy):.4f} / ROC-AUC={roc_auc_score(y_te, p_dummy):.4f}")
lr = make_pipeline(pre, LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
p_lr = lr.predict_proba(X_te)[:, 1]
print(f"[ロジスティック回帰] PR-AUC={average_precision_score(y_te, p_lr):.4f} / ROC-AUC={roc_auc_score(y_te, p_lr):.4f}")


[ダミー] PR-AUC=0.0192 / ROC-AUC=0.5000
[ロジスティック回帰] PR-AUC=0.1344 / ROC-AUC=0.8461


In [4]:
# 正解率は見ない。混同行列の実数と再現率を見る（閾値0.2は運用キャパシティの仮置き。調整ではない）
cm = confusion_matrix(y_te, (p_lr >= 0.2).astype(int))
print("混同行列（行=正解 0/1, 列=予測 0/1, 閾値0.2・ベースライン）:")
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"適合率: {tp / (tp + fp):.3f} / 再現率: {tp / (tp + fn):.3f}")
assert np.isfinite(p_lr).all()


混同行列（行=正解 0/1, 列=予測 0/1, 閾値0.2・ベースライン）:
[[4861   43]
 [  83   13]]
適合率: 0.232 / 再現率: 0.135


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
from sklearn.metrics import precision_recall_curve
plt.figure()
for name, p in [("dummy", p_dummy), ("logreg", p_lr)]:
    prec, rec, _ = precision_recall_curve(y_te, p)
    plt.step(rec, prec, where="post", label=name)
plt.xlabel("recall"); plt.ylabel("precision"); plt.legend(); plt.title("PR curves (test)")
plt.savefig(f"{FIG}/code-result-j02-01.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j02-01.png")


saved code-result-j02-01.png


## 読み取りどころ

- 不正率の低さに対し、正解率が99%前後でも無意味なこと。
- ダミーのPR-AUC≒不正率、線形のPR-AUCがベースラインの物差しになること（改善幅は `02_improved.ipynb` で測る）。
- 閾値0.2の再現率では見逃しが残ること。閾値運用は改善モデルのファイルへ。
